# Water Values for Long-Duration Storage Operation
This example demonstrates how the concept of water values (known from hydroelectric power plants, aka future cost function) can be used to capture seasonal behaviour of long-duration storage in a rolling horizon optimisation with limited operational foresight. Applying the concept of water values helps internalise the future value of stored energy beyond the optimisation horizon when making short-term operational decisions.

## Learning Content

## Code

In [1]:
import logging
import matplotlib.pyplot as plt
import pandas as pd

import pypsa

pypsa.options.params.optimize.solver_name = "gurobi"
logging.getLogger("gurobipy"). setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)

For this example, we consider a simple energy system with a single bus and 3-hourly time steps over one year (8760 hours in 2920 3-hour blocks). We have time-varying load, wind and solar geenration at zero marginal cost, a gas generator with amrginal cost function $C'(g) = 80+0.01g$ €/MWh, and hydrogen storage consisting of electrolysis, turbine, and a non-cyclic storage tank with a set initial energy. All components have fixed capacities; investments are not optimised here. 

In [2]:
n = pypsa.examples.model_energy()

n.remove("Generator", "load shedding")
n.remove("StorageUnit", "battery storage")

n.generators.loc[["solar", "wind"], "p_nom"] = 30_000, 20_000
n.generators.p_nom_extendable = False

n.links.loc[["electrolysis", "turbine"], "p_nom"] = 10_000, 20_000
n.links.p_nom_extendable = False

n.stores.loc["hydrogen storage", ["e_nom", "e_initial"]] = 2_000_000, 500_000
n.stores.e_nom_extendable = False
n.stores.e_cyclic = False

n.add("Carrier", "gas", color="darkorange")
n.add(
    "Generator",
    "gas",
    bus="electricity",
    p_nom = 40_000,
    marginal_cost = 80,
    marginal_cost_quadratic = 0.01,
    carrier="gas",
)


INFO:pypsa.network.io:Retrieving network data from https://github.com/PyPSA/PyPSA/raw/v1.0.7/examples/networks/model-energy/model-energy.nc.
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Model-Energy' has buses, carriers, generators, links, loads, storage_units, stores


### Case 1: Perfect Foresight

Let's first solve the problem with perfect foresight over the whole eyar to obtain a benchmark solution that captures the optimal seasonal operation of hydrogen storage.

In [3]:
n.optimize(assign_all_duals=True, log_to_console=False)

Writing continuous variables.: 100%|██████████| 4/4 [00:00<00:00, 263.33it/s]

Restricted license - for non-production use only - expires 2027-11-29
Read LP format model from file /tmp/linopy-problem-qsjomkrm.lp
Reading time = 0.05 seconds
obj: 43800 rows, 20440 columns, 67159 nonzeros
Set parameter LogToConsole to value 0


GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

Here, we get the following fuel costs (in M€/a):